# AQI Forecasting — Feature Engineering & Model Experiments

A learning-oriented, production-shaped notebook for forecasting AQI N days ahead
from pollutant + weather history. Every section explains **what** it does, **why**
it's structured that way, and **how** the code works — written so someone opening
this repo cold (including future-you) can follow it without side context.

**Design principles used throughout:**
- **Modular over hardcoded.** Nearly everything is a function that takes the
  forecast horizon, the split strategy, or the model as a parameter, instead of a
  notebook that only works for `aqi_next_1d` and breaks the moment you change it.
- **Every model goes through the same evaluation path.** One `evaluate_and_log()`
  and one `plot_model_diagnostics()` function are reused for every model, so
  results are directly comparable and you only have to learn to read one set of
  plots.
- **Delta-target throughout.** Every model predicts *change* in AQI
  (`aqi_next_Nd - european_aqi`), not the raw level — explained in Section 9.

1. Setup
2. API Client Layer — pass a city name, coordinates are resolved automatically
3. Data Validation Layer
4. Feature Engineering Layer
5. Quick Smoke Test
6. Historical Data Pipeline
7. Exploratory Data Analysis
8. Outlier Assessment & Handling
9. Target & Split Utilities — the modular horizon/split machinery
10. Model Training & Diagnostics Framework — one function, every model
11. Baseline Models: Random Forest & Ridge
12. XGBoost
13. Hyperparameter Tuning — GridSearchCV & RandomizedSearchCV
14. Walk-Forward Validation
15. Deep Learning: Feed-Forward NN & LSTM
16. Multi-Horizon Comparison (1-day / 2-day / 3-day ahead)
17. Model Leaderboard
18. Next Steps


# 02 — Feature Engineering & Transformation

Feature Engineering, Lag/Rolling transformations, and Target creation for Pearls AQI Predictor.


## 1. Setup

In [ ]:
!pip install openmeteo_requests requests_cache retry_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.4/230.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.9/809.9 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 111.1 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [ ]:
# --- stdlib ---
from abc import ABC, abstractmethod
from datetime import datetime
from typing import Any, Dict, List, Optional

# --- data & viz ---
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- API client ---
import openmeteo_requests
import requests_cache
from retry_requests import retry
from requests.exceptions import HTTPError

# --- validation ---
from pydantic import BaseModel, Field, field_validator, ValidationError

# --- sklearn ---
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error,
)

# --- xgboost ---
import xgboost as xgb

# --- deep learning ---
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

sns.set_theme(style="whitegrid")

# A running scoreboard. Every call to evaluate_and_log() (Section 10) appends one
# row here — this IS the notebook's source of truth for "which model won," so it
# gets reset at the top of every fresh run instead of silently accumulating rows
# across kernel sessions (that's an easy way to end up reporting stale numbers).
leaderboard = []

## 2. API Client Layer

**What this does:** wraps Open-Meteo's two APIs (air quality + historical weather)
behind one interface, and resolves a **city name** to coordinates internally so the
rest of the notebook never has to know or care about lat/lon.

**Why a city dictionary instead of a geocoding API call:** an extra network request
per city lookup is pure overhead for a fixed, known set of cities you're actually
going to use. A dictionary is one lookup, zero latency, zero failure mode. Add a
city once (`CITY_COORDINATES["your_city"] = (lat, lon)`) and every function below
picks it up automatically.

**Why an abstract base class for a single client:** `OpenMeteoClient` isn't the
only air-quality provider that will ever exist. Coding against `BaseAPIClient`
instead of `OpenMeteoClient` directly means adding a second provider later (a paid
fallback, a different region's better-covered source) is one new subclass, not a
rewrite of every place that calls `client.fetch_historical(...)`. This is the
**Strategy pattern**: the caller depends on the interface, not the implementation.

In [ ]:
# Known cities — add more as needed. Keys are lowercased for lookup;
# CITY_COORDINATES["lahore"] and CITY_COORDINATES["Lahore"] both resolve.
CITY_COORDINATES: Dict[str, tuple] = {
    "lahore": (31.558, 74.351),
    "karachi": (24.8607, 67.0011),
    "islamabad": (33.6844, 73.0479),
    "delhi": (28.6139, 77.2090),
    "beijing": (39.9042, 116.4074),
    "new york": (40.7128, -74.0060),
    "london": (51.5074, -0.1278),
    "los angeles": (34.0522, -118.2437),
}

def resolve_city(city: str) -> tuple:
    """Look up (lat, lon) for a known city name. Raises a clear error for an
    unknown one instead of silently failing deep inside an API call."""
    key = city.strip().lower()
    if key not in CITY_COORDINATES:
        known = ", ".join(sorted(CITY_COORDINATES))
        raise ValueError(f"Unknown city '{city}'. Known cities: {known}. "
                          f"Add it to CITY_COORDINATES to use it.")
    return CITY_COORDINATES[key]

In [ ]:
AQI_VARIABLES = [
    "pm10", "pm2_5", "carbon_monoxide", "nitrogen_dioxide",
    "sulphur_dioxide", "ozone", "uv_index", "aerosol_optical_depth", "european_aqi",
]

WEATHER_VARIABLES = [
    "temperature_2m", "wind_direction_10m", "wind_speed_10m", "rain",
    "weather_code", "wind_gusts_10m", "cloud_cover", "relative_humidity_2m",
]


class BaseAPIClient(ABC):
    """Abstract base for all API clients — every provider implements the same
    four methods, so the rest of the notebook only ever talks to this interface."""

    def __init__(self, api_key: str = None):
        self.api_key = api_key

    @abstractmethod
    def fetch_current(self, city: str) -> Dict[str, Any]:
        pass

    @abstractmethod
    def fetch_historical(self, city: str, start_date: str, end_date: str) -> list:
        pass

    @abstractmethod
    def fetch_current_weather(self, city: str) -> Dict[str, Any]:
        pass

    @abstractmethod
    def fetch_historical_weather(self, city: str, start_date: str, end_date: str) -> list:
        pass


class OpenMeteoClient(BaseAPIClient):
    """Open-Meteo air-quality + weather client. No API key needed."""

    AQI_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
    WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
    WEATHER_HISTORY_URL = "https://archive-api.open-meteo.com/v1/archive"

    def __init__(self):
        super().__init__()
        cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
        retry_session = retry(cache_session, retries=3, backoff_factor=0.2)
        self.client = openmeteo_requests.Client(session=retry_session)

    # ---- shared internals --------------------------------------------------

    def _fetch_current(self, url: str, variables: List[str], city: str,
                        lat: float, lon: float) -> Dict[str, Any]:
        params = {"latitude": lat, "longitude": lon, "current": variables, "timezone": "auto"}
        print(f"Fetching current data from {url} for {city} ({lat}, {lon})")
        try:
            response = self.client.weather_api(url, params=params)[0]
            current = response.Current()
            data = {
                "date": pd.to_datetime(current.Time(), unit="s", utc=True)
                    .tz_convert(response.Timezone().decode()).tz_localize(None),
                "city": city, "lat": lat, "lon": lon,
            }
            data.update({var: round(current.Variables(i).Value(), 5) for i, var in enumerate(variables)})
            return data
        except Exception as e:
            print(f"Request failed: {e}")
            raise HTTPError(f"OpenMeteo current request failed ({url}): {e}")

    def _fetch_historical(self, url: str, variables: List[str], city: str,
                           lat: float, lon: float, start_date: str, end_date: str) -> list:
        params = {
            "latitude": lat, "longitude": lon, "hourly": variables,
            "timezone": "auto", "start_date": start_date, "end_date": end_date,
        }
        print(f"Fetching historical data from {url} for {city} [{start_date} -> {end_date}]")
        try:
            response = self.client.weather_api(url, params=params)[0]
            hourly = response.Hourly()
            hourly_data = {
                "date": pd.date_range(
                    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=hourly.Interval()),
                    inclusive="left",
                ).tz_convert(response.Timezone().decode()),
                "city": city, "lat": lat, "lon": lon,
            }
            for i, var in enumerate(variables):
                hourly_data[var] = hourly.Variables(i).ValuesAsNumpy()
            return pd.DataFrame(hourly_data).to_dict(orient="records")
        except Exception as e:
            print(f"Request failed: {e}")
            raise HTTPError(f"OpenMeteo historical request failed ({url}): {e}")

    # ---- public API — all take a city NAME, coordinates resolved internally ---

    def fetch_current(self, city: str) -> Dict[str, Any]:
        lat, lon = resolve_city(city)
        return self._fetch_current(self.AQI_URL, AQI_VARIABLES, city, lat, lon)

    def fetch_historical(self, city: str, start_date: str, end_date: str) -> list:
        lat, lon = resolve_city(city)
        return self._fetch_historical(self.AQI_URL, AQI_VARIABLES, city, lat, lon, start_date, end_date)

    def fetch_current_weather(self, city: str) -> Dict[str, Any]:
        lat, lon = resolve_city(city)
        return self._fetch_current(self.WEATHER_URL, WEATHER_VARIABLES, city, lat, lon)

    def fetch_historical_weather(self, city: str, start_date: str, end_date: str) -> list:
        lat, lon = resolve_city(city)
        return self._fetch_historical(self.WEATHER_HISTORY_URL, WEATHER_VARIABLES, city, lat, lon, start_date, end_date)


class APIClientFactory:
    """Factory for the active client — swap providers here without touching
    anything that calls client.fetch_historical(...) elsewhere in the notebook."""

    @staticmethod
    def get_primary_client() -> BaseAPIClient:
        return OpenMeteoClient()

In [ ]:
client = APIClientFactory.get_primary_client()
CITY = "Lahore"   # change this one line to point the whole notebook at another city

## 3. Data Validation Layer

**What this does:** two checks, run at two different points in the pipeline.
`validate_raw_data` checks the API response *before* any feature engineering
touches it — wrong types, impossible values (negative concentrations, AQI outside
0-500), too many nulls. `validate_features` runs *after* feature engineering and
checks for **drift**: do the engineered features still look like what the model was
trained on, statistically?

**Why validate before AND after:** a bug in the API response and a bug in your own
feature engineering look identical downstream (a model that's suddenly wrong) but
need different fixes. Catching each at its own boundary tells you immediately which
side of the pipeline broke.

In [ ]:
class RawDataSchema(BaseModel):
    """Validates one row of raw API response before feature engineering."""
    date: datetime
    city: str
    lat: float
    lon: float
    temperature_2m: Optional[float] = Field(None)
    wind_direction_10m: Optional[float] = Field(None)
    wind_speed_10m: Optional[float] = Field(None)
    rain: Optional[float] = Field(None)
    weather_code: Optional[float] = Field(None)
    wind_gusts_10m: Optional[float] = Field(None)
    cloud_cover: Optional[float] = Field(None)
    relative_humidity_2m: Optional[float] = Field(None)
    pm10: Optional[float] = Field(None, ge=0)
    pm2_5: Optional[float] = Field(None, ge=0)
    carbon_monoxide: Optional[float] = Field(None, ge=0)
    nitrogen_dioxide: Optional[float] = Field(None, ge=0)
    sulphur_dioxide: Optional[float] = Field(None, ge=0)
    ozone: Optional[float] = Field(None, ge=0)
    uv_index: Optional[float] = Field(None, ge=0)
    aerosol_optical_depth: Optional[float] = Field(None, ge=0)
    european_aqi: float = Field(..., ge=0, le=500)

    @field_validator("european_aqi")
    @classmethod
    def aqi_must_be_reasonable(cls, v):
        if v > 300:
            print(f"Extremely high AQI detected: {v}")
        return v


class DataValidator:
    """Validates DataFrames against the schema above and a few business rules."""

    def validate_raw_data(self, df: pd.DataFrame) -> pd.DataFrame:
        validation_errors = []
        for idx, row in df.iterrows():
            try:
                RawDataSchema(**row.to_dict())
            except ValidationError as e:
                for err_dict in e.errors():
                    validation_errors.append({
                        **err_dict,
                        "loc": ("row", idx, *err_dict.get("loc", ())),
                        "input": row.to_dict(),
                    })
        if validation_errors:
            raise ValidationError.from_exception_data("RawDataSchema validation", validation_errors)

        null_rates = df.isnull().mean() * 100
        bad_cols = null_rates[null_rates > 5]
        if not bad_cols.empty:
            raise ValueError(f"Columns exceed 5% null rate: {bad_cols.to_dict()}")

        out_of_range = ~df["european_aqi"].between(0, 500)
        if out_of_range.any():
            raise ValueError(f"{out_of_range.sum()} row(s) with european_aqi out of [0, 500]")

        return df

    def build_reference_stats(self, train_df: pd.DataFrame) -> dict:
        """Snapshot of each numeric column's mean/std, taken from TRAINING data
        only. This is what validate_features() below compares new data against —
        it's the "what normal looks like" baseline."""
        columns = train_df.select_dtypes(include=["number"]).columns
        return {col: {"mean": train_df[col].mean(), "std": train_df[col].std()} for col in columns}

    def validate_features(self, df: pd.DataFrame, reference_stats: dict = None) -> pd.DataFrame:
        """Flags any column whose current mean has drifted more than 3 standard
        deviations (measured in the reference/training scale) from what training
        data looked like. A print, not a hard failure — drift is a signal to go
        look, not necessarily a broken pipeline (a real smog event SHOULD drift
        the pollutant columns; that's not a bug)."""
        if reference_stats is None:
            print("No reference stats provided — skipping drift check.")
            return df
        for col, stats in reference_stats.items():
            if col not in df.columns:
                continue
            current_mean = df[col].mean()
            if stats["std"] > 0:
                drift = abs(current_mean - stats["mean"]) / stats["std"]
                if drift > 3:
                    print(f"Feature drift detected in '{col}': current mean={current_mean:.2f}, reference mean={stats['mean']:.2f}")
        return df

## 4. Feature Engineering Layer

**What this does:** turns raw hourly readings into the feature set every model
below trains on — temporal encodings, pollutant ratios, lags, and rolling stats —
and builds the target column(s).

**Why `forecast_horizon` is a parameter, not a fixed column name:** we need "predict tomorrow" to "predict the next 3 days." Instead of
copy-pasting the feature engineering code once per horizon (and hand-editing three
near-identical copies every time something changes), `forecast_horizon=3` builds
`aqi_next_1d`, `aqi_next_2d`, and `aqi_next_3d` in one pass. Section 9 then picks
*one* of those columns as the active target — see the leakage note there for why
the others must be dropped from the feature set, not just ignored.

**A few specific choices worth understanding, not just copying:**
- **Cyclical (sin/cos) encoding for hour/day/month.** A model reading raw `hour=23`
  has no way to know it's adjacent to `hour=0` — to a linear model, 23 looks 23x
  "bigger" than 1. Mapping each cycle onto a circle (`sin`, `cos`) makes "close in
  time" mean "close in value" again.
- **`no2_o3_ratio` is clipped, not epsilon-divided.** `ozone` hits exactly 0 on
  hundreds of rows (mostly overnight). Dividing by `ozone + 1e-6` turns those rows
  into values in the hundreds of millions — a feature meant to range roughly 0-5
  spikes eight orders of magnitude past that, and standardizing it later doesn't
  fix the underlying distortion. Clipping the denominator to a physically sensible
  floor (5 µg/m³) keeps the ratio meaningful on every row instead of just moving
  where the outlier problem hides.
- **Lag/rolling windows include 24h.** The target is 24h out, so "what was AQI at
  this exact hour yesterday" (`aqi_lag_24h`) is a much more directly relevant
  signal than only looking at the last few hours.
- **Rolling stats are computed on `shift(1)` first.** Without that shift, a
  "rolling mean of the last 6 hours" would include the *current* hour — which
  means the feature partially contains the value you're trying to predict from it.
  That's leakage, and it's easy to miss because the model still trains and predicts
  fine; it just performs unrealistically well until deployed on genuinely unseen
  data.

In [ ]:
LAG_HOURS = [1, 3, 6, 24]
ROLLING_WINDOWS = [4, 6, 12, 24]


class AQIFeatureEngineer(BaseEstimator, TransformerMixin):
    """Feature engineering as an sklearn Transformer (fit/transform), so it can
    drop into a Pipeline later without special-casing."""

    def __init__(self, forecast_horizon: int = 1):
        self.forecast_horizon = forecast_horizon
        self.feature_names_ = None

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        df = df.sort_values(["city", "date"]).reset_index(drop=True)

        # 1. Temporal features
        df["hour"] = df["date"].dt.hour
        df["day_of_week"] = df["date"].dt.dayofweek
        df["month"] = df["date"].dt.month
        df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

        # 2. Cyclical encoding
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7.0)
        df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7.0)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df['wind_u'] = -df['wind_speed_10m'] * np.sin(np.radians(df['wind_direction_10m']))
        df['wind_v'] = -df['wind_speed_10m'] * np.cos(np.radians(df['wind_direction_10m']))

        # 3. Pollutant interaction features
        df["pm_ratio"] = df["pm2_5"] / (df["pm10"] + 1e-6)          # pm10 never near 0 — safe as-is
        df["no2_o3_ratio"] = df["nitrogen_dioxide"] / df["ozone"].clip(lower=5.0)

        # Continuous transition signals
        df['temp_trend_7d'] = df['temperature_2m'].rolling(24*7).mean().diff(24*7)  # is it warming/cooling fast?
        df['rain_accum_7d'] = df['rain'].rolling(24*7).sum()   # monsoon "arriving" shows up here

        g = df.groupby("city")["european_aqi"]

        # 4. Lag features
        for lag in LAG_HOURS:
            df[f"aqi_lag_{lag}h"] = g.shift(lag)

        # 5. Rolling statistics — shift(1) first, see the note above on why
        shifted = g.shift(1)
        for window in ROLLING_WINDOWS:
            df[f"aqi_roll_mean_{window}h"] = shifted.groupby(df["city"]).rolling(window).mean().reset_index(level=0, drop=True)
            df[f"aqi_roll_std_{window}h"] = shifted.groupby(df["city"]).rolling(window).std().reset_index(level=0, drop=True)

        # 6. Exponential rolling statistic — weights recent hours more than a flat window average
        df["aqi_ewm_6h"] = df["european_aqi"].ewm(span=6).mean()

        # 7. Rate of change
        df["aqi_change_1h"] = g.shift(1).diff(1)
        df["aqi_change_24h"] = g.shift(1).diff(24)

        # 8. Target creation — one column per day out to forecast_horizon
        for day in range(1, self.forecast_horizon + 1):
            df[f"aqi_next_{day}d"] = df.groupby("city")["european_aqi"].shift(-day * 24)

        self.feature_names_ = [c for c in df.columns if c not in ["date", "city"]]
        return df

    def get_feature_names_out(self, input_features=None) -> list:
        return self.feature_names_

## 9. Target & Split Utilities

This is the section that makes the rest of the notebook horizon-agnostic. Two
functions, both parameterized, used by everything downstream:

**`prepare_features_target(df, horizon)`** builds `X`, the delta target, the raw
target, and the "current AQI" series for **one specific horizon** (1, 2, or 3
days). It does one thing that matters a lot: when `forecast_horizon > 1`, the
feature-engineering step (Section 4) creates `aqi_next_1d`, `aqi_next_2d`, and
`aqi_next_3d` **all at once**, in the same dataframe. If you're training a model to
predict `aqi_next_2d` and you don't explicitly drop `aqi_next_1d` and
`aqi_next_3d` from `X`, you're handing the model *other future values* as input
features — a direct leakage bug, and the likely cause of the crash/nonsense
results from trying to predict `aqi_next_2d` last round. This function drops every
horizon column except the active target automatically, so that mistake can't
happen regardless of which horizon you're modeling.

**Why delta, not raw level:** `aqi_next_Nd - european_aqi` is the actual quantity
every model below is trained on. Predicting the raw AQI level directly fails
whenever the test window's typical AQI differs from training's (a tree literally
cannot predict below the minimum value it saw in training) — delta sidesteps that
by being centered near zero regardless of season. Reconstruction back to raw AQI
(`current_aqi + predicted_delta`) happens only at evaluation time.

**`make_split(...)`** returns either a single 80/20 holdout or a list of
`TimeSeriesSplit` folds, picked by one `method` argument — this is the "modular
between holdout and time-split" piece. **Time-split is the default** (`SPLIT_METHOD`
below) because a single holdout's score is highly sensitive to which season
happened to land in that one test window (demonstrated concretely in Section 14);
averaging across several folds gives a more honest number. Holdout stays available
because it's faster to iterate with while actively developing.

In [ ]:
BASE_DROP_COLS = ['date', 'city', 'lat', 'lon']
SPLIT_METHOD = "timeseries"   # "timeseries" (default) or "holdout" — change here, everything below follows


def prepare_features_target(df: pd.DataFrame, horizon: int) -> tuple:
    """Builds X, y_delta, y_raw, current_aqi for ONE forecast horizon.
    Drops every OTHER aqi_next_*d column so no future horizon leaks into features."""
    target_col = f"aqi_next_{horizon}d"
    if target_col not in df.columns:
        raise ValueError(f"'{target_col}' not found — was AQIFeatureEngineer built "
                          f"with forecast_horizon >= {horizon}?")

    d = df.dropna(subset=[target_col]).copy()
    d = d.dropna()
    d["target_delta"] = d[target_col] - d["european_aqi"]

    other_horizon_cols = [c for c in d.columns if c.startswith("aqi_next_") and c != target_col]
    drop_cols = BASE_DROP_COLS + [target_col, "target_delta"] + other_horizon_cols

    X = d.drop(columns=drop_cols)
    y_delta = d["target_delta"]
    y_raw = d[target_col]
    current_aqi = d["european_aqi"]
    return X, y_delta, y_raw, current_aqi


def make_holdout_split(X, y_delta, y_raw, current_aqi, test_frac: float = 0.2) -> dict:
    split_idx = int(len(X) * (1 - test_frac))
    return {
        "X_train": X.iloc[:split_idx], "X_test": X.iloc[split_idx:],
        "y_train_delta": y_delta.iloc[:split_idx], "y_test_delta": y_delta.iloc[split_idx:],
        "y_train_raw": y_raw.iloc[:split_idx], "y_test_raw": y_raw.iloc[split_idx:],
        "current_aqi_test": current_aqi.iloc[split_idx:],
    }


def make_timeseries_splits(X, y_delta, y_raw, current_aqi, n_splits: int = 5) -> list:
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []
    for train_idx, test_idx in tscv.split(X):
        folds.append({
            "X_train": X.iloc[train_idx], "X_test": X.iloc[test_idx],
            "y_train_delta": y_delta.iloc[train_idx], "y_test_delta": y_delta.iloc[test_idx],
            "y_train_raw": y_raw.iloc[train_idx], "y_test_raw": y_raw.iloc[test_idx],
            "current_aqi_test": current_aqi.iloc[test_idx],
        })
    return folds


def make_split(X, y_delta, y_raw, current_aqi, method: str = SPLIT_METHOD, **kwargs):
    """Single entry point — returns a dict for 'holdout', or a list of dicts
    (one per fold) for 'timeseries'. Downstream code checks which it got."""
    if method == "holdout":
        return make_holdout_split(X, y_delta, y_raw, current_aqi, **kwargs)
    elif method == "timeseries":
        return make_timeseries_splits(X, y_delta, y_raw, current_aqi, **kwargs)
    raise ValueError(f"Unknown split method: {method!r}")

In [ ]:
HORIZON = 1   # 1, 2, or 3 — change this and every section below retrains on that horizon

X, y_delta, y_raw, current_aqi = prepare_features_target(engineered_df, HORIZON)
split = make_holdout_split(X, y_delta, y_raw, current_aqi)   # holdout for the single-model sections below

X_train, X_test = split["X_train"], split["X_test"]
y_train_delta, y_test_delta = split["y_train_delta"], split["y_test_delta"]
y_train_raw, y_test_raw = split["y_train_raw"], split["y_test_raw"]
current_aqi_test = split["current_aqi_test"]

print(f"Horizon: {HORIZON}d ahead | train rows: {len(X_train)} | test rows: {len(X_test)}")

Horizon: 1d ahead | train rows: 24788 | test rows: 6197


In [ ]:
# Scaling for Ridge/NN/LSTM only — tree models (RF, XGBoost) train on the
# unscaled features directly, since splits are threshold-based and don't care
# about magnitude.
#
# RobustScaler, not StandardScaler: StandardScaler centers on the MEAN and scales
# by STANDARD DEVIATION, both of which the pollutant spikes found in Section 7-8
# drag around even after capping. RobustScaler centers on the MEDIAN and scales by
# the interquartile range — statistics that don't move much even with a heavy-
# tailed pollutant distribution still in the data. Given the outlier finding, this
# is the safer default for anything gradient- or distance-based.
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)